In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env file
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [ ]:
#tool createion
@tool
def get_conversion_factor(base_currency : str , target_currency : str) -> float:
    """Get the conversion factor between two currencies."""
    #this is dummy api url replace it with real one
    url = f"https://api.exchangerate-api.com/v4/latest/{base_currency}/{target_currency}"
    response = requests(url)

    return response.json()



@tool
def convert(base_currency : int , conversion_rate : Annotated[float, InjectedToolArg]) -> float:
    """Convert amount from base currency to target currency."""
    
    return base_currency * conversion_rate

In [ ]:
get_conversion_factor.invoke({'base_currency': 'USD','target_currency': 'PKR'})

In [ ]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo-0613", temperature=0.5)
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
messages = [HumanMessage("what is the conversion factor between USD and PKR? can u convert 10 UDD to PKR")]

In [ ]:
ai_message = llm_with_tools.invoke(messages)

In [ ]:
messages.append(ai_message)

In [ ]:
ai_message.tool_calls

In [ ]:
import json
for tool_call in ai_message.tool_calls:
    #execute the first tool and get the conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        #append it to message
        messages.append(tool_message1)
       #execute the second tool to convert 10 USD to PKR
    if tool_call['name'] == 'convert': 
        #fetch the current arguments
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)


In [ ]:
messages

In [ ]:
llm_with_tools.invoke(messages).content